# 25 — Competition & Relationship Features

**Objetivo:** caracterizar competencia observable del proceso antes de `t0`. Solo se cuentan participaciones con `publication_at <= t0`; una fecha faltante no se presume conocida.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.competition_features import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-25-competition-features-v1'

In [2]:
from argos.gold.competition_features import build_competition_features

result = build_competition_features(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

[25] materializing competition features from bidder observations available by t0...


'data/gold/features/competition_features.parquet'

In [3]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,competition_feature_grain,grain,True,True,"{""distinct_contracts"": 5640278, ""expected"": 56...",one competition-feature row per contract with ...
1,competition_features_pit_safe,point_in_time,True,True,"{""future_publication_violations"": 0}",no bidder publication used by a feature occurs...
2,competition_counts_nonnegative,features,True,True,"{""violations"": 0}",competition count features are non-negative


In [4]:
blocking = result.validation.loc[result.validation["blocking"]]
print("25 ✅ PASS" if result.ready_for_next_notebook else "25 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Competition features no superan su auditoría PIT."

25 ✅ PASS
Blocking checks: 3/3
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_competition_features.json
